# Dublin Bikes ML - Data Cleaning and Feature Selection

## Objective
The goal of this notebook is to prepare the merged Dublin Bikes and weather dataset for machine learning.

The target variable is:
- `num_bikes_available`


In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("final_merged_data.csv")
df.head()

,last_reported,station_id,num_bikes_available,num_docks_available,is_installed,is_renting,is_returning,name,address,lat,...,min_humidity_quality_indicator,min_relative_humidity_percent,humidity_std_quality_indicator,relative_humidity_std_deviation,max_pressure_quality_indicator,max_barometric_pressure_hpa,min_pressure_quality_indicator,min_barometric_pressure_hpa,pressure_std_quality_indicator,barometric_pressure_std_deviation
0,2024-12-01 00:10:00,10,15,1,True,True,True,DAME STREET,Dame Street,53.344006,...,0,83.2,0,0.284,0,1002.56,0,1002.26,0,0.083
1,2024-12-01 00:10:00,100,17,8,True,True,True,HEUSTON BRIDGE (SOUTH),Heuston Bridge (South),53.347107,...,0,83.2,0,0.284,0,1002.56,0,1002.26,0,0.083
2,2024-12-01 00:10:00,109,20,9,True,True,True,BUCKINGHAM STREET LOWER,Buckingham Street Lower,53.353333,...,0,83.2,0,0.284,0,1002.56,0,1002.26,0,0.083
3,2024-12-01 00:10:00,11,1,29,True,True,True,EARLSFORT TERRACE,Earlsfort Terrace,53.334293,...,0,83.2,0,0.284,0,1002.56,0,1002.26,0,0.083
4,2024-12-01 00:10:00,114,4,36,True,True,True,WILTON TERRACE (PARK),Wilton Terrace (Park),53.333652,...,0,83.2,0,0.284,0,1002.56,0,1002.26,0,0.083


## Initial dataset inspection

We first inspect the shape, columns, data types, and missing values in the merged dataset.

In [3]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (298946, 78)

Columns:
['last_reported', 'station_id', 'num_bikes_available', 'num_docks_available', 'is_installed', 'is_renting', 'is_returning', 'name', 'address', 'lat', 'lon', 'capacity', 'stno', 'year', 'month', 'day', 'hour', 'minute', 'max_air_temp_quality_indicator', 'max_air_temperature_celsius', 'min_air_temp_quality_indicator', 'min_air_temperature_celsius', 'air_temp_std_quality_indicator', 'air_temperature_std_deviation', 'max_grass_temp_quality_indicator', 'max_grass_temperature_celsius', 'min_grass_temp_quality_indicator', 'min_grass_temperature_celsius', 'grass_temp_std_quality_indicator', 'grass_temperature_std_deviation', 'max_soil_temp_5cm_quality_indicator', 'max_soil_temperature_5cm_celsius', 'min_soil_temp_5cm_quality_indicator', 'min_soil_temperature_5cm_celsius', 'soil_temp_std_5cm_quality_indicator', 'soil_temperature_std_deviation_5cm', 'max_soil_temp_10cm_quality_indicator', 'max_soil_temperature_10cm_celsius', 'min_soil_temp_10cm_quality_indicator', '

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 298946 entries, 0 to 298945
Data columns (total 78 columns):
 #   Column                                  Non-Null Count   Dtype  
---  ------                                  --------------   -----  
 0   last_reported                           298946 non-null  object 
 1   station_id                              298946 non-null  int64  
 2   num_bikes_available                     298946 non-null  int64  
 3   num_docks_available                     298946 non-null  int64  
 4   is_installed                            298946 non-null  bool   
 5   is_renting                              298946 non-null  bool   
 6   is_returning                            298946 non-null  bool   
 7   name                                    298946 non-null  object 
 8   address                                 298946 non-null  object 
 9   lat                                     298946 non-null  float64
 10  lon                                     2989

In [5]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
last_reported,298946,4433,2024-12-11 00:30:00,115,NaN,NaN,NaN,NaN,NaN,NaN,NaN
station_id,298946.0,NaN,NaN,NaN,57.967138,33.958176,1.0,28.0,57.0,88.0,117.0
num_bikes_available,298946.0,NaN,NaN,NaN,12.204733,9.761814,0.0,4.0,11.0,19.0,40.0
num_docks_available,298946.0,NaN,NaN,NaN,19.335198,11.001506,0.0,11.0,19.0,28.0,40.0
is_installed,298946,1,True,298946,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
max_barometric_pressure_hpa,298946.0,NaN,NaN,NaN,1014.85587,11.823993,975.06,1006.01,1017.21,1022.66,1035.82
min_pressure_quality_indicator,298946.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
min_barometric_pressure_hpa,298946.0,NaN,NaN,NaN,1014.689671,11.894264,975.01,1005.77,1017.06,1022.57,1035.73
pressure_std_quality_indicator,298946.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
missing = df.isnull().sum().sort_values(ascending=False)
missing[missing > 0]

Series([], dtype: int64)

## Missing value analysis

This step helps identify whether any important columns contain missing values that must be cleaned before modelling.

In [7]:
df["last_reported"] = pd.to_datetime(df["last_reported"], errors="coerce")

In [8]:
print("Missing timestamps after conversion:", df["last_reported"].isnull().sum())

Missing timestamps after conversion: 0


## Prediction target

The variable selected for prediction is:
- `num_bikes_available`

This is the most relevant target because it directly represents bike availability at each station.

In [9]:
target = "num_bikes_available"
print("Target column:", target)

Target column: num_bikes_available


## Removing irrelevant columns

Some columns are not suitable for modelling:
- text fields such as station name and address
- duplicate identifiers
- location coordinates for the baseline model
- quality indicator columns
- highly specialised weather sensor variables not directly relevant to bike demand

In [10]:
columns_to_drop = [
    "name",
    "address",
    "stno",
    "lat",
    "lon",
    "year",
    "month",
    "day",
    "minute"
]

In [11]:
quality_cols = [col for col in df.columns if "quality_indicator" in col]
len(quality_cols)

30

In [12]:
weather_noise_cols = [
    "max_grass_temperature_celsius",
    "min_grass_temperature_celsius",
    "grass_temperature_std_deviation",
    "max_soil_temperature_5cm_celsius",
    "min_soil_temperature_5cm_celsius",
    "soil_temperature_std_deviation_5cm",
    "max_soil_temperature_10cm_celsius",
    "min_soil_temperature_10cm_celsius",
    "soil_temperature_std_deviation_10cm",
    "max_soil_temperature_20cm_celsius",
    "min_soil_temperature_20cm_celsius",
    "soil_temperature_std_deviation_20cm",
    "max_earth_temperature_30cm_celsius",
    "min_earth_temperature_30cm_celsius",
    "earth_temperature_std_deviation_30cm",
    "max_earth_temperature_100cm_celsius",
    "min_earth_temperature_100cm_celsius",
    "earth_temperature_std_deviation_100cm",
]

In [13]:
drop_list = columns_to_drop + quality_cols + weather_noise_cols
df = df.drop(columns=drop_list, errors="ignore")

print("New shape:", df.shape)

New shape: (298946, 21)


## Feature engineering: time-based variables

Bike demand changes depending on:
- hour of day
- weekday vs weekend
- commuting peak hours

These features are derived from the timestamp.

In [14]:
df["day_of_week"] = df["last_reported"].dt.weekday
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
df["is_peak_hour"] = df["hour"].isin([7, 8, 9, 16, 17, 18]).astype(int)

In [15]:
df[["last_reported", "hour", "day_of_week", "is_weekend", "is_peak_hour"]].head()

,last_reported,hour,day_of_week,is_weekend,is_peak_hour
0,2024-12-01 00:10:00,0,6,1,0
1,2024-12-01 00:10:00,0,6,1,0
2,2024-12-01 00:10:00,0,6,1,0
3,2024-12-01 00:10:00,0,6,1,0
4,2024-12-01 00:10:00,0,6,1,0


## Feature engineering: simplified weather features

To reduce dimensionality and improve interpretability, average weather measures are created from daily minimum and maximum values.

In [16]:
df["avg_temp"] = (
    df["max_air_temperature_celsius"] + df["min_air_temperature_celsius"]
) / 2

df["avg_humidity"] = (
    df["max_relative_humidity_percent"] + df["min_relative_humidity_percent"]
) / 2

In [17]:
df[["avg_temp", "avg_humidity"]].head()

,avg_temp,avg_humidity
0,13.955,83.75
1,13.955,83.75
2,13.955,83.75
3,13.955,83.75
4,13.955,83.75


In [18]:
key_columns = [
    "station_id",
    "capacity",
    "hour",
    "day_of_week",
    "is_weekend",
    "is_peak_hour",
    "avg_temp",
    "avg_humidity",
    "max_barometric_pressure_hpa",
    target
]

df[key_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
station_id,298946.0,57.967138,33.958176,1.0000,28.0000,57.000,88.000,117.00
capacity,298946.0,31.979003,7.459101,16.0000,29.0000,30.000,40.000,40.00
hour,298946.0,11.522128,6.280459,0.0000,7.0000,12.000,16.000,23.00
day_of_week,298946.0,2.923478,2.060851,0.0000,1.0000,3.000,5.000,6.00
is_weekend,298946.0,0.282205,0.450073,0.0000,0.0000,0.000,1.000,1.00
is_peak_hour,298946.0,0.285396,0.451604,0.0000,0.0000,0.000,1.000,1.00
avg_temp,298946.0,7.785927,3.129450,-3.4845,5.6765,7.872,10.125,14.63
avg_humidity,298946.0,84.284922,8.004812,55.0700,79.7700,85.350,89.600,98.85
max_barometric_pressure_hpa,298946.0,1014.855870,11.823993,975.0600,1006.0100,1017.210,1022.660,1035.82
num_bikes_available,298946.0,12.204733,9.761814,0.0000,4.0000,11.000,19.000,40.00


## Cleaning missing values

Rows missing values in required modelling columns are removed to ensure the final dataset is valid for training.

In [19]:
required_columns = [
    "station_id",
    "capacity",
    "hour",
    "day_of_week",
    "is_weekend",
    "is_peak_hour",
    "avg_temp",
    "avg_humidity",
    "max_barometric_pressure_hpa",
    target
]

df_clean = df.dropna(subset=required_columns).copy()
print("Shape after dropping missing required values:", df_clean.shape)

Shape after dropping missing required values: (298946, 26)


In [20]:
print("Negative bikes:", (df_clean["num_bikes_available"] < 0).sum())
print("Bikes greater than capacity:", (df_clean["num_bikes_available"] > df_clean["capacity"]).sum())
print("Negative capacity:", (df_clean["capacity"] < 0).sum())

Negative bikes: 0
Bikes greater than capacity: 0
Negative capacity: 0


In [21]:
df_clean = df_clean[
    (df_clean["num_bikes_available"] >= 0) &
    (df_clean["capacity"] > 0) &
    (df_clean["num_bikes_available"] <= df_clean["capacity"])
].copy()

print("Shape after range validation:", df_clean.shape)

Shape after range validation: (298946, 26)


## Final selected features

The final feature set includes:
- time-based features
- station-level features
- weather-related features

`month` was excluded because the dataset spans only a single month and therefore has no useful variation.

In [22]:
final_features = [
    "station_id",
    "capacity",
    "hour",
    "day_of_week",
    "is_weekend",
    "is_peak_hour",
    "avg_temp",
    "avg_humidity",
    "max_barometric_pressure_hpa"
]

print(final_features)

['station_id', 'capacity', 'hour', 'day_of_week', 'is_weekend', 'is_peak_hour', 'avg_temp', 'avg_humidity', 'max_barometric_pressure_hpa']


In [23]:
final_df = df_clean[final_features + [target]].copy()
final_df.head()

,station_id,capacity,hour,day_of_week,is_weekend,is_peak_hour,avg_temp,avg_humidity,max_barometric_pressure_hpa,num_bikes_available
0,10,16,0,6,1,0,13.955,83.75,1002.56,15
1,100,25,0,6,1,0,13.955,83.75,1002.56,17
2,109,29,0,6,1,0,13.955,83.75,1002.56,20
3,11,30,0,6,1,0,13.955,83.75,1002.56,1
4,114,40,0,6,1,0,13.955,83.75,1002.56,4


In [24]:
final_df.corr(numeric_only=True)[target].sort_values(ascending=False)

num_bikes_available            1.000000
capacity                       0.205051
day_of_week                    0.010088
avg_humidity                   0.008658
is_weekend                     0.007177
avg_temp                       0.002011
station_id                    -0.001356
max_barometric_pressure_hpa   -0.002129
hour                          -0.004754
is_peak_hour                  -0.009573
Name: num_bikes_available, dtype: float64

## Save cleaned dataset

The final cleaned dataset is exported for use in model training.

In [25]:
final_df.to_csv("cleaned_ml_dataset.csv", index=False)
print("Saved as cleaned_ml_dataset.csv")

Saved as cleaned_ml_dataset.csv


## Summary

This notebook prepared the merged Dublin Bikes dataset for machine learning by:

- selecting `num_bikes_available` as the target variable
- removing irrelevant and noisy columns
- handling missing values
- deriving time-based features
- simplifying weather variables
- selecting a final set of modelling features
- exporting a cleaned dataset for model training

Final selected features:
- `station_id`
- `capacity`
- `hour`
- `day_of_week`
- `is_weekend`
- `is_peak_hour`
- `avg_temp`
- `avg_humidity`
- `max_barometric_pressure_hpa`

In [26]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [27]:
df = df.drop_duplicates().copy()
print("Shape after dropping duplicates:", df.shape)

Shape after dropping duplicates: (298946, 26)
